# train_density.py

**Updated for Desktop machine:** `/home/pun/Desktop`

**Description:** Train YOLOv11 on Density Detection Dataset - Updated paths for current machine

## Imports

In [1]:
#!/usr/bin/env python3
"""
Train YOLOv11 on Density Variant Dataset
Updated for /home/pun/Desktop
"""
from ultralytics import YOLO
import os
import torch
import yaml

print("="*70)
print("🚀 YOLOv11 Training on Density Dataset")
print("="*70)

🚀 YOLOv11 Training on Density Dataset


## Configuration

In [ ]:
# Paths configuration
BASE_DIR = "/home/pun/Desktop"
DATA_YAML = f"{BASE_DIR}/yolov11/dataset/data_density.yaml"
PARAMS_YAML = f"{BASE_DIR}/notebooks_density/training/runs/detect/density_tune4/best_hyperparameters.yaml"

# Training configuration
MODEL_NAME = 'yolo11n.pt'  # Nano model (fastest)
EPOCHS = 100
IMG_SIZE = 320
BATCH_SIZE = 8
PROJECT_NAME = 'density_training'

# Device configuration - auto-detect GPU
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print("✅ GPU (CUDA) detected - using GPU acceleration")
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    print("✅ Apple Silicon (MPS) detected - using MPS acceleration")
else:
    DEVICE = 'cpu'
    print("⚠️  No GPU detected - using CPU (training will be slower)")

print(f"\n📁 Dataset config: {DATA_YAML}")
print(f"🔧 Device: {DEVICE}")
print(f"🏋️  Epochs: {EPOCHS}")
print(f"📏 Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"📦 Batch size: {BATCH_SIZE}")
print("="*70)

✅ GPU (CUDA) detected - using GPU acceleration

📁 Dataset config: /home/pun/Desktop/yolov11/dataset/data_density.yaml
🔧 Device: cuda
🏋️  Epochs: 100
📏 Image size: 320x320
📦 Batch size: 8


## Set default parameters

In [3]:
default_hyp = {
    # Augmentation parameters - optimized for density detection
    "degrees": 20.0,      # Rotation augmentation
    "translate": 0.15,    # Translation augmentation
    "scale": 0.7,         # Scale augmentation
    "flipud": 0.5,        # Vertical flip probability
    "fliplr": 0.5,        # Horizontal flip probability
    "mosaic": 1.0,        # Mosaic augmentation (4 images combined)
    "mixup": 0.1,         # Mixup augmentation (blend 2 images)

    # Preserve grayscale/density values
    "hsv_h": 0.0,         # No hue change (grayscale)
    "hsv_s": 0.0,         # No saturation change
    "hsv_v": 0.0,         # No brightness change (preserve density values)

    # Optimization parameters
    "lr0": 0.01,          # Initial learning rate
    "lrf": 0.01,          # Final learning rate factor
    "momentum": 0.937,    # SGD momentum
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0, # Warmup epochs
    "close_mosaic": 10,   # Disable mosaic 10 epochs before end
}

## Verify Dataset

In [4]:
# Verify dataset exists
if not os.path.exists(DATA_YAML):
    print(f"❌ ERROR: Dataset config not found at {DATA_YAML}")
    print("Please make sure the yolov11/dataset folder is in the correct location.")
else:
    print(f"✅ Dataset config found: {DATA_YAML}")
    
    # Check if images exist
    train_dir = f"{BASE_DIR}/yolov11/dataset/images/train/density"
    val_dir = f"{BASE_DIR}/yolov11/dataset/images/val/density"
    test_dir = f"{BASE_DIR}/yolov11/dataset/images/test/density"
    
    train_count = len([f for f in os.listdir(train_dir) if f.endswith('.jpg')]) if os.path.exists(train_dir) else 0
    val_count = len([f for f in os.listdir(val_dir) if f.endswith('.jpg')]) if os.path.exists(val_dir) else 0
    test_count = len([f for f in os.listdir(test_dir) if f.endswith('.jpg')]) if os.path.exists(test_dir) else 0
    
    print(f"\n📊 Dataset Statistics:")
    print(f"   Training images: {train_count}")
    print(f"   Validation images: {val_count}")
    print(f"   Test images: {test_count}")
    print(f"   Total images: {train_count + val_count + test_count}")

# Load parameters from previous hyperparameter tuning
print(f"\n🔄 Loading hyperparameters from: {PARAMS_YAML}")
with open(PARAMS_YAML, 'r') as f:
    try:
        best_hyp = yaml.safe_load(f)
        print("✅ Hyperparameters loaded successfully:")
    except yaml.YAMLError as exc:
        print(f"❌ Error loading hyperparameters: {exc}")
        best_hyp = default_hyp  # Default hyperparameters if loading fails

✅ Dataset config found: /home/pun/Desktop/yolov11/dataset/data_density.yaml

📊 Dataset Statistics:
   Training images: 192
   Validation images: 24
   Test images: 72
   Total images: 288

🔄 Loading hyperparameters from: /home/pun/Desktop/notebooks_density/training/runs/detect/density_tune4/best_hyperparameters2.yaml
✅ Hyperparameters loaded successfully:


## Initialize Model

In [5]:
# Initialize YOLO model - will auto-download if not present
print("\n🔧 Initializing YOLO model...")
model = YOLO(MODEL_NAME)
print(f"✅ Model loaded: {MODEL_NAME}")


🔧 Initializing YOLO model...
✅ Model loaded: yolo11n.pt


## Start Training

In [6]:
# Train the model
print("\n" + "="*70)
print("🏋️  Starting Training...")
print("="*70 + "\n")

results = model.train(
    # Dataset
    data=DATA_YAML,
    
    # Training parameters
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    
    # Output
    name=PROJECT_NAME,
    project='runs/detect',
    
    # Device
    device=DEVICE,
    workers=4,
    
    # Early stopping
    patience=30,  # Stop if no improvement for 30 epochs
    
    # Output options
    save=True,
    plots=True,
    verbose=True,

    **best_hyp  # Unpack best hyperparameters
)


🏋️  Starting Training...

Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24202MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=11, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/pun/Desktop/yolov11/dataset/data_density.yaml, degrees=4.14134, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.13679, flipud=0.33633, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00952, lrf=0.02, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.69311, mosaic=0.68334, multi_scale=False, name=density_training8, nbs=64, nms=False, opset=None, optimiz

## Training Results

In [7]:
print(f"\n{'='*70}")
print("✅ Training Complete!")
print(f"{'='*70}")
print(f"\n📁 Results saved to: runs/detect/{PROJECT_NAME}/")
print(f"🏆 Best weights: runs/detect/{PROJECT_NAME}/weights/best.pt")
print(f"📊 Last weights: runs/detect/{PROJECT_NAME}/weights/last.pt")
print(f"\n📈 Training metrics and plots saved in the results folder.")
print(f"{'='*70}")


✅ Training Complete!

📁 Results saved to: runs/detect/density_training/
🏆 Best weights: runs/detect/density_training/weights/best.pt
📊 Last weights: runs/detect/density_training/weights/last.pt

📈 Training metrics and plots saved in the results folder.


## Validation (Optional)

In [8]:
# Validate the best model
print("\n" + "="*70)
print("📊 Running Validation on Best Model...")
print("="*70 + "\n")

best_model = YOLO(f'runs/detect/{PROJECT_NAME}/weights/best.pt')
metrics = best_model.val()

print("\n📊 Validation Metrics:")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")


📊 Running Validation on Best Model...

Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24202MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1103.5±176.8 MB/s, size: 17.5 KB)
val: Scanning /home/pun/Desktop/yolov11/dataset/labels/val/density.cache... 24 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 24/24 36.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.2it/s 1.7s0.7s
                   all         24       1992      0.979      0.983      0.992       0.73
Speed: 1.3ms preprocess, 35.9ms inference, 0.0ms loss, 3.6ms postprocess per image
Results saved to /home/pun/Desktop/notebooks_density/training/runs/detect/val8

📊 Validation Metrics:
   Precision: 0.9787
   Recall: 0.9834
   mAP50: 0.9924
   mAP50-95: 0.7298
